# 01 Effective Area 可視化

2種類のビューを提供する。

| ビュー | 内容 |
| --- | --- |
| **フレーム詳細ビュー** | 1フレームの三角形採用状況を3パネルで確認する |
| **時系列ビュー** | 試合全体のEA面積推移を折れ線グラフで確認する |

## 設定

試合を変える場合は `MATCH_DATE` / `MATCH_ID` / `MATCH_LABEL` を変更する。
時系列の処理速度を変えたい場合は `SAMPLE_EVERY_N_FRAMES` を調整する（大きくするほど速い）。

In [ ]:
import ast
import xml.etree.ElementTree as ET
from itertools import combinations
from pathlib import Path

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from shapely.geometry import MultiPolygon, Polygon
from shapely.ops import unary_union

PROJECT_ROOT = (
    Path.cwd().parent.parent if Path.cwd().name in ('重心', 'effective_area')
    else Path.cwd().parent if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

MATCH_DATE  = '2023-11-25'
MATCH_ID    = '117093'
MATCH_LABEL = 'tsukuba_vs_tsukuba-b'
DATASET_NAME = f'{MATCH_DATE}_{MATCH_ID}_{MATCH_LABEL}'

DATASET_DIR  = PROJECT_ROOT / 'data' / 'candidates' / '01_sorted' / DATASET_NAME
TRACKING_XML = DATASET_DIR / '01_tracking_core' / f'{MATCH_ID}_tracker_box_data.xml'
METADATA_XML = DATASET_DIR / '03_player_master' / f'{MATCH_ID}_tracker_box_metadata.xml'
OUTPUT_DIR   = PROJECT_ROOT / 'outputs' / 'tables' / DATASET_NAME
FIGURE_DIR   = PROJECT_ROOT / 'outputs' / 'figures' / DATASET_NAME / 'effective_area'

PERIMETER_THRESHOLD  = 36.0   # meters: 守備EAの周囲長閾値（これを超える場合、守備EAと重なる攻撃三角形を除外）
INCLUDE_GK           = True   # GK含む11人（論文通り）
VALID_BALL_STATUSES  = ('HOME', 'AWAY')
SAMPLE_EVERY_N_FRAMES = 25    # 時系列: 有効フレームを N 枚おきにサンプリング

print(DATASET_NAME)
print('tracking XML:', TRACKING_XML.exists())
print('metadata XML:', METADATA_XML.exists())

---

## 関数定義

以下のセルを**上から順にすべて実行**してから、フレーム詳細ビュー・時系列ビューを使う。

| グループ | 関数 | 役割 |
| --- | --- | --- |
| データ読み込み | `load_tracking_metadata` | XMLからピッチ・チーム・選手情報を取得 |
| データ読み込み | `load_player_positions_by_frame` | XMLから全フレームの選手座標を取得 |
| EA計算 | `_perimeter` | 三角形の周囲長を計算 |
| EA計算 | `classify_defensive_triangles` | 守備三角形を採用/非採用に分類 |
| EA計算 | `classify_attacking_triangles` | 攻撃三角形を採用/非採用に分類 |
| EA計算 | `compute_effective_area` | 守備EA・攻撃EAをまとめて計算 |
| EA計算 | `process_all_frames` | 複数フレームを一括処理してDataFrameを返す |
| 可視化 | `_draw_pitch` | ピッチの下地を描画 |
| 可視化 | `_draw_geom` | Shapely ジオメトリを塗りつぶしで描画 |
| 可視化 | `_draw_triangles` | 三角形リストを個別に描画 |
| 可視化 | `plot_triangle_breakdown` | 三角形採用状況を3パネルで表示 |
| 可視化 | `plot_ea_timeseries` | EA面積の時系列グラフを表示 |

### データ読み込み

#### `load_tracking_metadata(metadata_xml_path)`

metadata XML を読み込み、ピッチサイズ・チーム・選手の情報を返す。

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `metadata_xml_path` | Path / str | metadata XMLのパス |

| 戻り値のキー | 型 | 内容 |
| --- | --- | --- |
| `pitch` | dict | ピッチサイズ `{width: 105.0, height: 68.0}`（メートル） |
| `teams` | DataFrame | `team_id / team_name / side / team_label` |
| `players` | DataFrame | `player_id / player_name / position / team_id / side / team_label` |

> **チームラベルの規約**: `side='away'` → ラベル `A`、`side='home'` → ラベル `B`

In [ ]:
def load_tracking_metadata(metadata_xml_path):
    root = ET.parse(metadata_xml_path).getroot()

    pitch = root.find('pitch')
    pitch_info = {'width': float(pitch.attrib['width']), 'height': float(pitch.attrib['height'])}

    teams = []
    for team in root.findall('.//teams/team'):
        teams.append({
            'team_id':    str(team.attrib['id']),
            'team_name':  team.attrib['name'],
            'side':       team.attrib['side'],
            'team_label': 'A' if team.attrib['side'] == 'away' else 'B',
        })

    players = []
    for player in root.findall('.//players/player'):
        players.append({
            'player_id':   str(player.attrib['id']),
            'player_name': player.attrib.get('name', ''),
            'position':    player.attrib.get('position', ''),
            'team_id':     str(player.attrib['teamId']),
        })

    teams_df   = pd.DataFrame(teams)
    players_df = pd.DataFrame(players).merge(
        teams_df[['team_id', 'side', 'team_label']], on='team_id', how='left')
    return {'pitch': pitch_info, 'teams': teams_df, 'players': players_df}

#### `load_player_positions_by_frame(tracking_xml_path, metadata, include_gk=True)`

tracking XML から全フレームの選手座標を読み込む。座標は **メートル単位** に変換して返す。

```
座標変換:
    XML の値は 0〜1 の正規化座標
    x_m = x_raw × pitch_width   例: 0.28 × 105 = 29.4 m
    y_m = y_raw × pitch_height  例: 0.53 × 68  = 36.0 m
```

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `tracking_xml_path` | Path / str | tracking XMLのパス |
| `metadata` | dict | `load_tracking_metadata` の戻り値 |
| `include_gk` | bool | `True` = GK含む11人（論文の定義）|

| 戻り値 | 型 | 内容 |
| --- | --- | --- |
| （リスト） | list of dict | 1要素 = 1フレーム |

各フレームの dict のキー:

| キー | 型 | 内容 |
| --- | --- | --- |
| `frame_number` | int | フレーム番号 |
| `match_time` | int | 試合時間（ミリ秒） |
| `event_period` | str | `'FIRST_HALF'` または `'SECOND_HALF'` |
| `ball_status` | str | `'HOME'` / `'AWAY'` / `'BALLOUT'` / `'NEUTRAL'` |
| `team_positions` | dict | `{team_id: [(x_m, y_m), ...]}` |

In [ ]:
def load_player_positions_by_frame(tracking_xml_path, metadata, include_gk=True):
    players_by_id = metadata['players'].set_index('player_id').to_dict('index')
    pitch_w = metadata['pitch']['width']
    pitch_h = metadata['pitch']['height']

    frames = []
    for _, elem in ET.iterparse(str(tracking_xml_path), events=('end',)):
        if elem.tag != 'frame':
            continue

        team_positions = {}
        for player in elem.findall('player'):
            player_id   = str(player.attrib['playerId'])
            player_info = players_by_id.get(player_id)
            if not player_info:
                continue
            if not include_gk and player_info.get('position') == 'GK':
                continue

            team_id    = str(player_info['team_id'])
            raw_x, raw_y = ast.literal_eval(player.attrib['loc'])
            team_positions.setdefault(team_id, []).append(
                (float(raw_x) * pitch_w, float(raw_y) * pitch_h))

        frames.append({
            'frame_number': int(float(elem.attrib['frameNumber'])),
            'match_time':   int(float(elem.attrib['matchTime'])),
            'event_period': elem.attrib.get('eventPeriod', ''),
            'ball_status':  elem.attrib.get('ballStatus', ''),
            'team_positions': team_positions,
        })
        elem.clear()  # 処理済み要素を破棄してメモリを節約

    return frames

### EA 計算

論文 **Definition 4（表面積指標 2）** のアルゴリズム。

```
① 守備 EA
   11人の選手座標から 11C3 = 165 個の三角形を生成
   → 全三角形の和集合 = 守備 EA（周囲長によるフィルタなし）

② 攻撃 EA
   同様に 165 個の三角形を生成
   → 各三角形を順に評価し、以下のいずれかに該当すれば除外:
      (1) 守備EAの周囲長 > 36m  かつ  守備EAと重なる
      (2) 採用済み攻撃EAと重なる
   → 除外されなかった三角形の和集合 = 攻撃 EA
```

#### `_perimeter(p1, p2, p3)`

3頂点からなる三角形の**周囲長（メートル）**を返す。
各引数は `(x, y)` のタプル（メートル座標）。

In [ ]:
def _perimeter(p1, p2, p3):
    d = lambda a, b: np.sqrt((a[0] - b[0]) ** 2 + (a[1] - b[1]) ** 2)
    return d(p1, p2) + d(p2, p3) + d(p3, p1)

#### `classify_defensive_triangles(player_positions)`

守備チームの全三角形（165個）を生成し、すべてを **採用** として返す。

守備 EA は全三角形の和集合として定義される（周囲長によるフィルタはなし）。

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `player_positions` | list of (x, y) | 守備チームの選手座標（メートル） |

| 戻り値 | 型 | 内容 |
| --- | --- | --- |
| `adopted` | list of Polygon | 全有効三角形（165個から縮退を除いたもの） |
| `rejected` | list of Polygon | 常に空リスト（守備EAに除外条件なし） |

In [ ]:
def classify_defensive_triangles(player_positions):
    adopted = []
    for i, j, k in combinations(range(len(player_positions)), 3):
        tri = Polygon([player_positions[i], player_positions[j], player_positions[k]])
        if tri.is_valid and tri.area > 1e-9:
            adopted.append(tri)
    return adopted, []  # 守備三角形に除外条件なし: rejected は常に空

#### `classify_attacking_triangles(player_positions, defensive_ea, perimeter_threshold=36.0)`

攻撃チームの全三角形（165個）を **採用 / 非採用** に分類する。

**除外条件（どちらかに該当したら除外）:**
1. 守備 EA の周囲長 > `perimeter_threshold` **かつ** 守備 EA と重なる
2. 採用済み攻撃 EA と重なる

三角形は `itertools.combinations` の生成順（選手インデックス昇順）で評価する。
採用されると、次の三角形の評価時に occupied に加算される（逐次的に広がる）。

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `player_positions` | list of (x, y) | 攻撃チームの選手座標（メートル） |
| `defensive_ea` | Shapely geometry | 守備 EA（`classify_defensive_triangles` の結果の和集合） |
| `perimeter_threshold` | float | 守備EAの周囲長閾値（メートル）。これを超えると守備EA重なり判定が有効になる |

| 戻り値 | 型 | 内容 |
| --- | --- | --- |
| `adopted` | list of Polygon | 採用された三角形 |
| `rejected` | list of Polygon | 非採用の三角形 |
| `attacking_ea` | Shapely geometry | 採用三角形の和集合（= 攻撃 EA） |

In [ ]:
def classify_attacking_triangles(player_positions, defensive_ea, perimeter_threshold=36.0):
    def_large    = defensive_ea.length > perimeter_threshold  # フレームごとに1回だけ計算
    attacking_ea = Polygon()
    adopted, rejected = [], []
    for i, j, k in combinations(range(len(player_positions)), 3):
        p1, p2, p3 = player_positions[i], player_positions[j], player_positions[k]
        tri = Polygon([p1, p2, p3])
        if not tri.is_valid or tri.area < 1e-9:
            continue
        # 採用済み攻撃EAと重なる → 除外
        if attacking_ea.intersection(tri).area >= 1e-9:
            rejected.append(tri)
            continue
        # 守備EAが大きく（周囲長 > 閾値）かつ守備EAと重なる → 除外
        if def_large and defensive_ea.intersection(tri).area >= 1e-9:
            rejected.append(tri)
            continue
        attacking_ea = attacking_ea.union(tri)
        adopted.append(tri)
    return adopted, rejected, attacking_ea

#### `compute_effective_area(def_positions, att_positions, perimeter_threshold=36.0)`

守備・攻撃 EA を **まとめて計算** するラッパー関数。
内部で `classify_defensive_triangles` → `classify_attacking_triangles` の順に呼ぶ。

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `def_positions` | list of (x, y) | 守備チームの選手座標（メートル） |
| `att_positions` | list of (x, y) | 攻撃チームの選手座標（メートル） |
| `perimeter_threshold` | float | 守備EAの周囲長閾値（メートル）。これを超えると守備EA重なり判定が有効になる |

| 戻り値のキー | 型 | 内容 |
| --- | --- | --- |
| `defensive_ea` | Shapely geometry | 守備 EA |
| `attacking_ea` | Shapely geometry | 攻撃 EA |
| `defensive_area_m2` | float | 守備 EA 面積（m²） |
| `attacking_area_m2` | float | 攻撃 EA 面積（m²） |
| `n_defensive_triangles` | int | 採用した守備三角形数 |
| `n_attacking_triangles` | int | 採用した攻撃三角形数 |

In [ ]:
def compute_effective_area(def_positions, att_positions, perimeter_threshold=36.0):
    def_adopted, _ = classify_defensive_triangles(def_positions)
    defensive_ea   = unary_union(def_adopted) if def_adopted else Polygon()
    att_adopted, _, attacking_ea = classify_attacking_triangles(att_positions, defensive_ea, perimeter_threshold)
    return {
        'defensive_ea':          defensive_ea,
        'attacking_ea':          attacking_ea,
        'defensive_area_m2':     defensive_ea.area,
        'attacking_area_m2':     attacking_ea.area,
        'n_defensive_triangles': len(def_adopted),
        'n_attacking_triangles': len(att_adopted),
    }

#### `process_all_frames(frames, metadata, perimeter_threshold=36.0)`

フレームリストを受け取り、各フレームの EA を計算して **DataFrame** で返す。

`ball_status` が `HOME` または `AWAY` のフレームのみ処理する。`BALLOUT` / `NEUTRAL` は除外する。

| 引数 | 型 | 説明 |
| --- | --- | --- |
| `frames` | list of dict | `load_player_positions_by_frame` の戻り値 |
| `metadata` | dict | `load_tracking_metadata` の戻り値 |
| `perimeter_threshold` | float | 守備三角形の採用閾値 |

戻り値の DataFrame の列:

| 列名 | 内容 |
| --- | --- |
| `frame_number` | フレーム番号 |
| `match_time_s` | 試合時間（秒） |
| `event_period` | 前半 / 後半 |
| `ball_status` | HOME / AWAY |
| `attacking_team_label` | 攻撃チームのラベル（A または B） |
| `defending_team_label` | 守備チームのラベル |
| `defensive_ea_m2` | 守備 EA 面積（m²） |
| `attacking_ea_m2` | 攻撃 EA 面積（m²） |
| `n_defensive_triangles` | 採用した守備三角形数 |
| `n_attacking_triangles` | 採用した攻撃三角形数 |

In [ ]:
def process_all_frames(frames, metadata, perimeter_threshold=36.0):
    teams_by_side = metadata['teams'].set_index('side')
    home_id       = teams_by_side.loc['home', 'team_id']
    away_id       = teams_by_side.loc['away', 'team_id']
    teams_by_id   = metadata['teams'].set_index('team_id')

    rows = []
    for frame in frames:
        bs = frame['ball_status']
        if bs not in ('HOME', 'AWAY'):
            continue
        pos = frame['team_positions']
        if home_id not in pos or away_id not in pos:
            continue

        att_id = home_id if bs == 'HOME' else away_id
        def_id = away_id if bs == 'HOME' else home_id
        result    = compute_effective_area(pos[def_id], pos[att_id], perimeter_threshold)
        att_label = teams_by_id.loc[att_id, 'team_label'] if att_id in teams_by_id.index else '?'
        def_label = teams_by_id.loc[def_id, 'team_label'] if def_id in teams_by_id.index else '?'

        rows.append({
            'frame_number':          frame['frame_number'],
            'match_time_s':          frame['match_time'] / 1000,
            'event_period':          frame['event_period'],
            'ball_status':           bs,
            'attacking_team_label':  att_label,
            'defending_team_label':  def_label,
            'n_att_players':         len(pos[att_id]),
            'n_def_players':         len(pos[def_id]),
            'defensive_ea_m2':       result['defensive_area_m2'],
            'attacking_ea_m2':       result['attacking_area_m2'],
            'n_defensive_triangles': result['n_defensive_triangles'],
            'n_attacking_triangles': result['n_attacking_triangles'],
        })

    return pd.DataFrame(rows)

### 可視化

#### `_draw_pitch(ax, pitch_w, pitch_h)`

ピッチの下地（芝・中央線・センターサークル）を描画する。
他の描画関数を呼ぶ**前に必ず呼ぶ**こと。

In [ ]:
def _draw_pitch(ax, pitch_w, pitch_h):
    ax.set_facecolor('#3d7a4a')
    ax.add_patch(plt.Rectangle(
        (0, 0), pitch_w, pitch_h, facecolor='#4a8c56', edgecolor='white', linewidth=2))
    ax.plot([pitch_w / 2, pitch_w / 2], [0, pitch_h], 'white', linewidth=1)
    ax.add_patch(mpatches.Circle(
        (pitch_w / 2, pitch_h / 2), 9.15, fill=False, color='white', linewidth=1))
    ax.set_xlim(-2, pitch_w + 2)
    ax.set_ylim(-2, pitch_h + 2)
    ax.set_aspect('equal')
    ax.set_xlabel('x (m)', fontsize=9)
    ax.set_ylabel('y (m)', fontsize=9)

#### `_draw_geom(ax, geom, facecolor, edgecolor, alpha, linewidth=1.0)`

Shapely の **Polygon または MultiPolygon** を塗りつぶしで描画する。
`unary_union` の結果が複数の飛び地（MultiPolygon）になるケースにも対応している。

In [ ]:
def _draw_geom(ax, geom, facecolor, edgecolor, alpha, linewidth=1.0):
    if geom.is_empty:
        return
    geoms = list(geom.geoms) if isinstance(geom, MultiPolygon) else [geom]
    for poly in geoms:
        x, y = poly.exterior.xy
        ax.fill(x, y, facecolor=facecolor, edgecolor=edgecolor,
                alpha=alpha, linewidth=linewidth)

#### `_draw_triangles(ax, triangles, facecolor, edgecolor, face_alpha, edge_alpha)`

三角形リストを**個別に**描画する。
採用三角形（色あり）と非採用三角形（薄い枠のみ）を描き分けることで、
どこにどの三角形があるかを視覚的に比較できる。

| 引数 | 説明 |
| --- | --- |
| `face_alpha` | 塗りつぶしの透明度（0 = 完全透明、1 = 不透明） |
| `edge_alpha` | 枠線の透明度 |

In [ ]:
def _draw_triangles(ax, triangles, facecolor, edgecolor, face_alpha, edge_alpha):
    for tri in triangles:
        x, y = tri.exterior.xy
        ax.fill(x, y, facecolor=facecolor, alpha=face_alpha, linewidth=0)
        ax.plot(list(x), list(y), color=edgecolor, alpha=edge_alpha, linewidth=0.6)

#### `plot_triangle_breakdown(...)`

1フレームの三角形採用状況を **3パネル** で可視化する。

| パネル | 表示内容 |
| --- | --- |
| **左** | 守備三角形。青塗り = 全三角形採用（フィルタなし）。タイトルに守備EA周囲長と重なり判定の有効/無効を表示 |
| **中** | 攻撃三角形。赤塗り = 採用、灰色枠 = 非採用。守備EAを薄く参考表示 |
| **右** | 最終 EA のみ表示（守備: 青、攻撃: 赤） |

`save_path` を指定すると PNG を保存する。

In [ ]:
def plot_triangle_breakdown(frame_data, metadata, def_positions, att_positions,
                             def_adopted, def_rejected, att_adopted, att_rejected,
                             def_ea, att_ea, def_team_id, att_team_id,
                             perimeter_threshold=36.0, save_path=None):
    pitch_w   = metadata['pitch']['width']
    pitch_h   = metadata['pitch']['height']
    teams_idx = metadata['teams'].set_index('team_id')
    def_name  = teams_idx.loc[def_team_id, 'team_name'] if def_team_id in teams_idx.index else def_team_id
    att_name  = teams_idx.loc[att_team_id, 'team_name'] if att_team_id in teams_idx.index else att_team_id
    def_large = def_ea.length > perimeter_threshold

    fig, axes = plt.subplots(1, 3, figsize=(36, 10))

    # --- 左: 守備三角形（全採用） ---
    ax = axes[0]
    _draw_pitch(ax, pitch_w, pitch_h)
    _draw_triangles(ax, def_adopted, '#3366cc', '#3366cc', face_alpha=0.06, edge_alpha=0.5)
    _draw_geom(ax, def_ea, '#3366cc', '#1144aa', alpha=0.45)
    for x, y in def_positions:
        ax.plot(x, y, 'o', color='#3366cc', markersize=9, markeredgecolor='white', markeredgewidth=1.2)
    ax.legend(handles=[
        mpatches.Patch(facecolor='#3366cc', alpha=0.7,
                       label=f'全三角形採用: {len(def_adopted)}個'),
    ], loc='upper right', fontsize=9)
    ax.set_title(
        f'{def_name}（守備）\n守備EA: {def_ea.area:.1f} m²'
        f'\n守備EA周囲長: {def_ea.length:.1f} m'
        f'  →  {">" if def_large else "≤"} {perimeter_threshold}m'
        f'  →  攻撃重なり判定: {"有効" if def_large else "無効"}',
        fontsize=10)

    # --- 中: 攻撃三角形 ---
    ax = axes[1]
    _draw_pitch(ax, pitch_w, pitch_h)
    _draw_geom(ax, def_ea, '#3366cc', '#1144aa', alpha=0.2)
    _draw_triangles(ax, att_rejected, 'gray',    'gray',    face_alpha=0.0,  edge_alpha=0.2)
    _draw_triangles(ax, att_adopted,  '#cc3333', '#cc3333', face_alpha=0.08, edge_alpha=0.5)
    _draw_geom(ax, att_ea, '#cc3333', '#aa1111', alpha=0.45)
    for x, y in att_positions:
        ax.plot(x, y, 'o', color='#cc3333', markersize=9, markeredgecolor='white', markeredgewidth=1.2)
    ax.legend(handles=[
        mpatches.Patch(facecolor='#cc3333', alpha=0.7,
                       label=f'採用: {len(att_adopted)}個'),
        mpatches.Patch(facecolor='gray',    alpha=0.4,
                       label=f'非採用: {len(att_rejected)}個'),
        mpatches.Patch(facecolor='#3366cc', alpha=0.3, label='守備EA（参考）'),
    ], loc='upper right', fontsize=9)
    ax.set_title(f'{att_name}（攻撃）\n攻撃EA: {att_ea.area:.1f} m²', fontsize=11)

    # --- 右: 最終EA ---
    ax = axes[2]
    _draw_pitch(ax, pitch_w, pitch_h)
    _draw_geom(ax, def_ea, '#3366cc', '#1144aa', alpha=0.45)
    _draw_geom(ax, att_ea, '#cc3333', '#aa1111', alpha=0.45)
    for x, y in def_positions:
        ax.plot(x, y, 'o', color='#3366cc', markersize=9, markeredgecolor='white', markeredgewidth=1.2)
    for x, y in att_positions:
        ax.plot(x, y, 'o', color='#cc3333', markersize=9, markeredgecolor='white', markeredgewidth=1.2)
    ax.legend(handles=[
        mpatches.Patch(facecolor='#3366cc', alpha=0.6,
                       label=f'{def_name}（守備）EA: {def_ea.area:.1f} m²'),
        mpatches.Patch(facecolor='#cc3333', alpha=0.6,
                       label=f'{att_name}（攻撃）EA: {att_ea.area:.1f} m²'),
    ], loc='upper right', fontsize=9)
    ax.set_title(
        f'最終 Effective Area\n'
        f'Frame {frame_data["frame_number"]} ({frame_data["match_time"] / 1000:.1f}s)',
        fontsize=11)

    fig.suptitle(
        f'Effective Area 三角形ビュー'
        f' — {frame_data["event_period"]}, ballStatus={frame_data["ball_status"]}',
        fontsize=13, y=1.01)
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()
    return fig, axes

#### `plot_ea_timeseries(ea_df, metadata, save_path=None)`

試合全体の EA 面積推移を **折れ線グラフ** で表示する。

| パネル | 内容 |
| --- | --- |
| **上** | チーム A が攻撃中のフレームにおける守備EA・攻撃EA |
| **下** | チーム B が攻撃中のフレームにおける守備EA・攻撃EA |

前半・後半の境界には点線を引く。`save_path` を指定すると PNG を保存する。

In [ ]:
def plot_ea_timeseries(ea_df, metadata, save_path=None):
    teams_df = metadata['teams'].set_index('team_label')

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
    fig.subplots_adjust(hspace=0.35)

    for ax_idx, att_label in enumerate(['A', 'B']):
        ax        = axes[ax_idx]
        def_label = 'B' if att_label == 'A' else 'A'
        df        = ea_df[ea_df['attacking_team_label'] == att_label].sort_values('match_time_s')
        att_name  = teams_df.loc[att_label, 'team_name'] if att_label in teams_df.index else att_label
        def_name  = teams_df.loc[def_label, 'team_name'] if def_label in teams_df.index else def_label
        att_color = '#cc3333' if att_label == 'A' else '#3366cc'
        def_color = '#3366cc' if att_label == 'A' else '#cc3333'

        ax.plot(df['match_time_s'], df['attacking_ea_m2'],
                color=att_color, linewidth=1.5, label=f'{att_name}（攻撃）EA')
        ax.plot(df['match_time_s'], df['defensive_ea_m2'],
                color=def_color, linewidth=1.5, linestyle='--', label=f'{def_name}（守備）EA')

        if 'event_period' in df.columns:
            second_half = df[df['event_period'] == 'SECOND_HALF']['match_time_s']
            if not second_half.empty:
                ax.axvline(second_half.min(), color='white', linewidth=1,
                           linestyle=':', alpha=0.6, label='後半開始')

        ax.set_ylabel('EA 面積 (m\u00b2)', fontsize=10)
        ax.set_facecolor('#1a1a2e')
        ax.tick_params(colors='white')
        ax.yaxis.label.set_color('white')
        ax.spines[:].set_color('#444')
        ax.set_title(f'{att_name} が攻撃中（{len(df)} フレーム）', fontsize=10, color='white')
        ax.legend(loc='upper right', fontsize=9, facecolor='#2a2a3e', labelcolor='white')
        ax.grid(axis='y', color='#444', linewidth=0.5)

    axes[-1].set_xlabel('試合時間 (s)', fontsize=10, color='white')
    fig.patch.set_facecolor('#0d0d1a')
    fig.suptitle('Effective Area 時系列', fontsize=13, color='white')
    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    plt.show()
    return fig, axes

---

## データ読み込み

In [ ]:
metadata = load_tracking_metadata(METADATA_XML)
print('Pitch:', metadata['pitch'])
print()

print('Loading tracking data...')
frames = load_player_positions_by_frame(TRACKING_XML, metadata, include_gk=INCLUDE_GK)
valid_frames = [f for f in frames if f['ball_status'] in VALID_BALL_STATUSES]
print(f'Total frames: {len(frames):,}  /  Valid frames: {len(valid_frames):,}')

---

## フレーム詳細ビュー

`DEMO_FRAME_INDEX` で見たいフレームを指定する（0 = 最初の有効フレーム）。

In [ ]:
DEMO_FRAME_INDEX = 0  # ← ここを変えて別フレームを確認できる

demo_frame   = valid_frames[DEMO_FRAME_INDEX]
teams_side   = metadata['teams'].set_index('side')
home_team_id = teams_side.loc['home', 'team_id']
away_team_id = teams_side.loc['away', 'team_id']

bs          = demo_frame['ball_status']
att_team_id = home_team_id if bs == 'HOME' else away_team_id
def_team_id = away_team_id if bs == 'HOME' else home_team_id
att_positions = demo_frame['team_positions'].get(att_team_id, [])
def_positions = demo_frame['team_positions'].get(def_team_id, [])

print(f'Frame {demo_frame["frame_number"]} — {demo_frame["event_period"]},'
      f' {demo_frame["match_time"] / 1000:.1f}s, ballStatus={bs}')
print(f'Defending: {def_team_id} ({len(def_positions)} players)')
print(f'Attacking: {att_team_id} ({len(att_positions)} players)')

In [ ]:
def_adopted, def_rejected = classify_defensive_triangles(def_positions)
def_ea = unary_union(def_adopted) if def_adopted else Polygon()
att_adopted, att_rejected, att_ea = classify_attacking_triangles(att_positions, def_ea, PERIMETER_THRESHOLD)

print('=== 守備 EA ===')
print(f'  採用三角形 : {len(def_adopted)} / 165')
print(f'  面積       : {def_ea.area:.2f} m²')
print(f'  周囲長     : {def_ea.length:.2f} m  ({">" if def_ea.length > PERIMETER_THRESHOLD else "≤"} {PERIMETER_THRESHOLD}m → 攻撃重なり判定: {"有効" if def_ea.length > PERIMETER_THRESHOLD else "無効"})')
print()
print('=== 攻撃 EA ===')
print(f'  採用三角形 : {len(att_adopted)} / 165')
print(f'  面積       : {att_ea.area:.2f} m²')

In [ ]:
fig, axes = plot_triangle_breakdown(
    demo_frame, metadata, def_positions, att_positions,
    def_adopted, def_rejected, att_adopted, att_rejected,
    def_ea, att_ea, def_team_id, att_team_id,
    perimeter_threshold=PERIMETER_THRESHOLD,
    save_path=FIGURE_DIR / f'breakdown_frame_{demo_frame["frame_number"]}.png',
)

---

## 時系列ビュー

試合全体の EA 面積推移を確認する。

`SAMPLE_EVERY_N_FRAMES` フレームおきにサンプリングして計算する。
処理時間の目安: 100フレームで約1〜2分。
速くしたい場合は `SAMPLE_EVERY_N_FRAMES` を大きくする（例: 50 → 約半分の時間）。

In [ ]:
sampled = [f for i, f in enumerate(valid_frames) if i % SAMPLE_EVERY_N_FRAMES == 0]
print(f'Processing {len(sampled)} frames  '
      f'(1/{SAMPLE_EVERY_N_FRAMES} of {len(valid_frames)} valid frames)...')

ea_df = process_all_frames(sampled, metadata, PERIMETER_THRESHOLD)
print(f'Done. {len(ea_df)} rows')
ea_df.head()

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / 'effective_area.csv'
ea_df.to_csv(output_path, index=False)
print(f'Saved: {output_path}')

In [ ]:
fig, axes = plot_ea_timeseries(
    ea_df, metadata,
    save_path=FIGURE_DIR / 'ea_timeseries.png',
)

In [ ]:
print('=== EA 統計まとめ ===')
ea_df.groupby('attacking_team_label').agg(
    frames           = ('frame_number',   'count'),
    mean_def_ea_m2   = ('defensive_ea_m2', 'mean'),
    mean_att_ea_m2   = ('attacking_ea_m2', 'mean'),
    median_att_ea_m2 = ('attacking_ea_m2', 'median'),
    std_att_ea_m2    = ('attacking_ea_m2', 'std'),
).round(2)